# Iris Flower Classification

**Project 1 of the ML/DL for Beginners series**

## The Problem

We have measurements of 150 iris flowers — sepal length, sepal width, petal length, and petal width — and each flower belongs to one of 3 species: **Setosa**, **Versicolor**, or **Virginica**.

Our goal: build a model that can predict a flower's species just from its measurements.

## Why This Dataset?

This is the "Hello World" of machine learning for good reason:
- It's small (150 rows) and clean — no missing values, no messy formatting
- It's a **classification problem** — one of the two most common ML task types (the other being regression)
- The 3 species are visually and numerically distinct, so even simple models do well — great for building confidence early

By the end of this notebook, you'll have trained your first machine learning model and understand *why* each step exists, not just *what* code makes it run.

## Step 1: Import Libraries

Here's what each library does:

- **pandas** — lets us work with data in table form (like a spreadsheet, but in code)
- **numpy** — handles numerical operations under the hood
- **scikit-learn (sklearn)** — the main machine learning library we'll use for the model, data splitting, and evaluation
- **matplotlib / seaborn** — for visualizing the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_style("whitegrid")

## Step 2: Load & Explore the Data

The Iris dataset is built directly into scikit-learn, so we don't need to download anything. We'll load it and immediately turn it into a pandas DataFrame — this makes it much easier to look at and visualize.

**Why explore before modeling?** Skipping this step is one of the most common beginner mistakes. You want to know: What does the data look like? Are there any obvious patterns? Are the classes balanced? Exploring first helps you catch problems before you waste time training a model on bad data.

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

df.head()

Let's check some basic info: how many rows, any missing values, and summary statistics for each measurement.

In [ ]:
print("Shape of dataset:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nSummary statistics:")
df.describe()

Now let's check that the 3 species are evenly represented — an imbalanced dataset (e.g. 140 of one species and 5 of another) would need special handling, which we'll cover in a later project.

In [ ]:
df['species'].value_counts()

**Visualizing the data.** A pairplot shows every measurement plotted against every other measurement, colored by species. This is one of the fastest ways to see whether your features actually separate the classes — if the colors form distinct clusters, a model should do well.

In [ ]:
sns.pairplot(df, hue='species', diag_kind='hist')
plt.suptitle("Iris Measurements by Species", y=1.02)
plt.show()

**What to notice:** Setosa (one color) is almost completely separated from the other two species in most plots — especially using petal length and petal width. Versicolor and Virginica overlap a bit more, which means the model will likely find those two species slightly harder to distinguish than Setosa.

## Step 3: Data Cleaning

We already saw above that there are **no missing values** in this dataset — that's why the `isnull().sum()` output was all zeros.

This step still matters even when the answer is "nothing to clean" — checking should always happen, never assumed. In later projects (like Titanic), this step will involve actual decisions about how to handle missing data.

## Step 4: Train/Test Split

**Why split the data?** If we train and test our model on the exact same data, it could just "memorize" the answers rather than actually learning the underlying pattern — this is called **overfitting**. To get an honest measure of how well the model performs on data it hasn't seen, we hold out a portion of the data (the "test set") that the model never sees during training.

We'll use 80% of the data for training and 20% for testing.

In [ ]:
X = df[iris.feature_names]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

**Note on `random_state=42`:** this just makes the split reproducible — anyone running this notebook will get the exact same train/test split, which makes results comparable.

## Step 5: Model Building

We'll use **K-Nearest Neighbors (KNN)** — one of the simplest classification algorithms to understand:

> To classify a new flower, KNN looks at the `k` closest flowers in the training data (by measurement similarity) and predicts whatever species is most common among those neighbors.

For example, with `k=5`, it looks at the 5 most similar flowers already seen and takes a majority vote.

We'll start with `k=5` as a reasonable default.

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

print("Model trained!")

## Step 6: Model Evaluation

Now we use the model to predict species for the **test set** — data it has never seen — and check how many it gets right.

**Accuracy** = (number of correct predictions) / (total predictions). It's the simplest metric, but it's not always the full picture (more on that in later projects when classes are imbalanced).

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

**Confusion matrix.** This shows exactly which species get confused with which. Rows are the actual species, columns are what the model predicted. Perfect predictions would show numbers only on the diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=iris.target_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

**Reading this chart:** if you see any off-diagonal numbers, those are the misclassifications — and given what we saw in the pairplot earlier (Versicolor and Virginica overlapping a bit), any mistakes are likely to be between those two species, not involving Setosa.

## Step 7: Conclusion

You just completed the full machine learning workflow:

1. Loaded and explored a dataset
2. Checked (and confirmed) data quality
3. Split data into training and testing sets
4. Trained a K-Nearest Neighbors classifier
5. Evaluated it using accuracy and a confusion matrix

This exact workflow — load, explore, clean, split, train, evaluate — is the backbone of almost every ML project you'll do, no matter how complex the dataset or model gets. The next few projects in this series will each add one new piece: handling missing data, working with imbalanced classes, and eventually moving into deep learning.

## 🚀 Try These Next

- Change `n_neighbors` in the model to 1, then to 15. Does accuracy go up, down, or stay the same? Why do you think that happens?
- Replace `KNeighborsClassifier` with `from sklearn.linear_model import LogisticRegression` — does it perform differently?
- Try dropping one column from `X` (e.g. drop petal width) before splitting — does accuracy drop? Which feature seems most important based on what you saw in the pairplot?

**Next project in the series:** Titanic Survival Prediction — where you'll deal with real messy data, missing values, and your first real feature engineering decisions.